In [1]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

### Path("knowledge-base") 
Returns a Path object (concretely a PosixPath on Mac/Linux or WindowsPath on Windows). It’s not a string and not a file handle. It’s a path object that represents the location knowledge-base, and it’s relative to whatever directory your script or notebook is running from.

#### Key points:

Creating it does not touch the disk. The folder doesn’t have to exist, and no check happens yet.
It gives you methods for working with that location: .exists(), .is_dir(), .iterdir(), .glob("*.md"), .read_text(), and so on.
It supports / for joining: KNOWLEDGE_BASE_PATH / "notes.md" gives Path("knowledge-base/notes.md").
If you print it, you see knowledge-base. In a notebook cell, the repr is PosixPath('knowledge-base').

In [6]:
load_dotenv(override=True)
MODEL = "gpt-4.1-nano"
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = "500"

openai = OpenAI()

In [ ]:
class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text, metadata=metadata)

class Chunks(BaseModel):
    chunks: list(Chunk)

### KNOWLEDGE_BASE_PATH.iterdir() 

KNOWLEDGE_BASE_PATH.iterdir() returns a generator object, not a list. It lazily yields one Path object for each item directly inside that folder, both files and subfolders, in arbitrary order.

Nothing has been read yet. To see the contents, consume the generator.

In [ ]:
def fetch_documents():
    """ A self coded version of function similar to LangChain DirectoryLoader """
    documents = [] # List to hold list of documents
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.glob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type,"source": file.as_posix(), "text": f.read()})
    print(f"Loaded {len(documents)} documents")
    return documents


In [8]:
check_file_folder()

products False
contracts False
company False
employees False
